# ML Zoomcamp 2026 — Homework 3: Classification
Official data: https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

Run cells in order. Requires pandas, NumPy and scikit-learn. Downloads the CSV if absent. No feature scaling is applied. Test data is held out; all model comparisons use validation data.

In [1]:
import pandas as pd
import numpy as np
import sklearn
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
path = Path("course_lead_scoring_2026.csv")
if not path.exists():
    urlretrieve(url, path)
print("scikit-learn:", sklearn.__version__)
print("CSV SHA256:", hashlib.sha256(path.read_bytes()).hexdigest())
df = pd.read_csv(path)
categorical = df.select_dtypes(include=["object", "string"]).columns.tolist()
numerical = [c for c in df.select_dtypes(include="number").columns if c != "converted"]
print("Shape:", df.shape)
print("Missing before filling:\n", df.isna().sum())
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)
assert not df.isna().any().any()
print("Categorical:", categorical)
print("Numerical:", numerical)

scikit-learn: 1.9.1
CSV SHA256: 2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9
Shape: (5000, 9)
Missing before filling:
 lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


## Q1 — Mode of industry
Compute after filling categorical missing entries with NA.

In [2]:
print(df.industry.value_counts())
q1 = df.industry.mode().iloc[0]
print("Q1:", q1)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Q1: technology


## Q2 — Correlation
Compute on the prepared numeric features. Compare only the four listed pairs.

In [3]:
corr = df[numerical].corr()
print(corr.to_string())
pairs = [("interaction_count", "lead_score"),
         ("number_of_courses_viewed", "lead_score"),
         ("number_of_courses_viewed", "interaction_count"),
         ("annual_income", "interaction_count")]
pair_scores = {pair: float(corr.loc[pair[0], pair[1]]) for pair in pairs}
for pair, score in pair_scores.items():
    print(pair, score)
q2 = max(pair_scores, key=pair_scores.get)
print("Q2:", q2)

                          annual_income  number_of_courses_viewed  interaction_count  lead_score
annual_income                  1.000000                  0.161300           0.122842    0.229496
number_of_courses_viewed       0.161300                  1.000000           0.721609    0.757204
interaction_count              0.122842                  0.721609           1.000000    0.915746
lead_score                     0.229496                  0.757204           0.915746    1.000000
('interaction_count', 'lead_score') 0.9157457980418697
('number_of_courses_viewed', 'lead_score') 0.7572042392104886
('number_of_courses_viewed', 'interaction_count') 0.7216090038937035
('annual_income', 'interaction_count') 0.12284235135955959
Q2: ('interaction_count', 'lead_score')


## Exact train / validation / test split
Extract target arrays and remove converted from every feature frame to prevent leakage.

In [4]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
y_train = df_train.converted.to_numpy()
y_val = df_val.converted.to_numpy()
y_test = df_test.converted.to_numpy()
df_train = df_train.drop(columns="converted").reset_index(drop=True)
df_val = df_val.drop(columns="converted").reset_index(drop=True)
df_test = df_test.drop(columns="converted").reset_index(drop=True)
df_full_train = df_full_train.drop(columns="converted").reset_index(drop=True)
print("Split:", len(df_train), len(df_val), len(df_test))
assert "converted" not in df_train.columns

Split: 3000 1000 1000


## Q3 — Mutual information
Use training data only. Compare scores rounded to two decimals.

In [5]:
mi = {col: mutual_info_score(y_train, df_train[col]) for col in categorical}
for col, score in mi.items():
    print(col, score, "rounded:", round(score, 2))
q3 = max(mi, key=lambda c: round(mi[c], 2))
print("Q3:", q3)

lead_source 0.03434605791617973 rounded: 0.03
industry 0.0019945636715935217 rounded: 0.0
employment_status 0.018965604906982875 rounded: 0.02
location 0.001042763579701525 rounded: 0.0
Q3: lead_source


## Q4 — Logistic regression
DictVectorizer learns categorical one-hot encoding on training data only. Numeric columns remain numeric. Use the prescribed parameters and the default 0.5 prediction threshold.

In [6]:
all_features = categorical + numerical

def fit_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    score = accuracy_score(y_val, model.predict(X_val))
    return float(score)

baseline = fit_and_score(all_features)
q4 = round(baseline, 2)
print("Baseline accuracy:", baseline)
print("Q4:", q4)

Baseline accuracy: 0.645
Q4: 0.65


## Q5 — Feature elimination
Drop each listed original feature, fit a fresh encoder/model, and compute baseline minus reduced-model accuracy without rounding. A negative difference means accuracy improved after removing that feature.

In [7]:
elimination = {}
for feature in ["lead_source", "number_of_courses_viewed", "interaction_count"]:
    reduced = [c for c in all_features if c != feature]
    score = fit_and_score(reduced)
    difference = baseline - score
    elimination[feature] = difference
    print(feature, "accuracy:", score, "difference:", difference)
q5 = min(elimination, key=elimination.get)
print("Q5:", q5)

lead_source accuracy: 0.642 difference: 0.0030000000000000027
number_of_courses_viewed accuracy: 0.643 difference: 0.0020000000000000018
interaction_count accuracy: 0.601 difference: 0.04400000000000004
Q5: number_of_courses_viewed


## Q6 — Regularization
Compare validation accuracy rounded to three decimals. Choose the smallest C in a tie. Smaller C means stronger regularization.

In [8]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {C: fit_and_score(all_features, C) for C in c_values}
for C, score in c_scores.items():
    print("C:", C, "accuracy:", score, "rounded:", round(score, 3))
best_score = max(round(score, 3) for score in c_scores.values())
q6 = min(C for C, score in c_scores.items() if round(score, 3) == best_score)
print("Q6:", q6)
print("ANSWERS:", {"Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4, "Q5": q5, "Q6": q6})

C: 1e-06 accuracy: 0.598 rounded: 0.598
C: 1e-05 accuracy: 0.598 rounded: 0.598
C: 0.0001 accuracy: 0.613 rounded: 0.613
C: 0.001 accuracy: 0.645 rounded: 0.645
Q6: 0.001
ANSWERS: {'Q1': 'technology', 'Q2': ('interaction_count', 'lead_score'), 'Q3': 'lead_source', 'Q4': 0.65, 'Q5': 'number_of_courses_viewed', 'Q6': 0.001}
